In [12]:

# Standard library imports
import json
import os
from typing import Any, Dict, List, Optional

BASE_PATH = "/kaggle/input/datasets/premshaw23/mmfundus-text"
BRSET_FILE = os.path.join(BASE_PATH, "BRSET_16266.json")

print("BRSET path:", BRSET_FILE)
print("File exists:", os.path.exists(BRSET_FILE))


BRSET path: /kaggle/input/datasets/premshaw23/mmfundus-text/BRSET_16266.json
File exists: True


In [13]:

# Load BRSET clinical metadata

if not os.path.exists(BRSET_FILE):
    raise FileNotFoundError(
        f"BRSET file not found at: {BRSET_FILE}\n"
        "Check that the MMFundus Text dataset is attached to the Kaggle notebook."
    )

with open(BRSET_FILE, "r", encoding="utf-8") as f:
    brset = json.load(f)

if not isinstance(brset, list) or len(brset) == 0:
    raise ValueError("BRSET_16266.json was loaded, but it does not contain a non-empty list.")

print(f"Loaded records: {len(brset):,}")
print("Available fields in first record:")
print(sorted(brset[0].keys()))


Loaded records: 16,266
Available fields in first record:
['AV_label', 'Answer', 'Answer0', 'Answer1', 'DR_ICDR', 'DR_SDRG', 'FBC_label', 'FHE_label', 'ImageID', 'Instruction', 'Instruction0', 'Instruction1', 'Keyword', 'Macular_label', 'OCD_label', 'Other_label', 'amd', 'artifacts', 'camera', 'comorbidities', 'diabetes', 'diabetes_time_y', 'diabetic_retinopathy', 'drusens', 'exam_eye', 'focus', 'hemorrhage', 'hypertensive_retinopathy', 'iluminaton', 'image_field', 'increased_cup_disc', 'insuline', 'macula', 'macular_edema', 'myopic_fundus', 'nationality', 'nevus', 'optic_disc', 'other', 'patient_age', 'patient_id', 'patient_sex', 'quality', 'retinal_detachment', 'scar', 'vascular_occlusion', 'vessels']


In [14]:

def _to_float(value: Any) -> Optional[float]:
    """Convert an explicitly supplied numeric value to float; otherwise return None."""
    if value is None:
        return None

    if isinstance(value, str):
        value = value.strip()
        if value == "" or value.lower() in {"nan", "none", "null"}:
            return None

    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def _clean_optional(value: Any) -> Any:
    """Return None for empty/missing values without inventing a replacement."""
    if value is None:
        return None

    if isinstance(value, str):
        value = value.strip()
        if value == "" or value.lower() in {"nan", "none", "null"}:
            return None

    return value


In [15]:

def clinical_note_agent(record: Dict[str, Any]) -> Dict[str, Any]:
    """
    Convert one BRSET clinical metadata record into RetinaAgent's
    structured Clinical-Note Agent output.

    No unsupported clinical value is inferred or fabricated.

    Parameters
    ----------
    record : dict
        One record from BRSET_16266.json.

    Returns
    -------
    dict
        Structured clinical context for the downstream Grader Agent.
    """

    if not isinstance(record, dict):
        raise TypeError("record must be a dictionary.")

    # Explicitly available in BRSET.
    diabetes_duration = _to_float(record.get("diabetes_time_y"))
    image_quality = _clean_optional(record.get("quality"))

    image_id = record.get("ImageID")
    if image_id is None:
        raise KeyError("BRSET record is missing the required 'ImageID' field.")

    return {
        # Pipeline bookkeeping / case identifier, not a clinical feature.
        "image_id": image_id,

        # Paper entity: visual acuity.
        # BRSET does not provide right/left visual-acuity values here.
        "visual_acuity": {
            "right_eye": None,
            "left_eye": None,
        },

        # Paper entity: lens status.
        "lens_status": None,

        # Paper entity: prior laser / anti-VEGF.
        # Kept separate for a convenient downstream schema.
        "prior_laser": None,
        "prior_anti_vegf": None,

        # Paper entity: HbA1c.
        "hba1c": None,

        # Paper entity: diabetes duration.
        "diabetes_duration": diabetes_duration,

        # Paper entity: prior vitrectomy.
        "prior_vitrectomy": None,

        # Paper entity: symptoms.
        "symptoms": [],

        # Paper entity: image-quality comment.
        "image_quality": image_quality,

        # Explicitly track fields that were unavailable rather than hiding
        # the missing-data regime.
        "missing_fields": [
            "visual_acuity",
            "lens_status",
            "prior_laser",
            "prior_anti_vegf",
            "hba1c",
            "prior_vitrectomy",
            "symptoms",
        ],
    }


In [16]:

record = brset[0]

note_output = clinical_note_agent(record)

print(json.dumps(note_output, indent=2, default=str))


{
  "image_id": "/All_data/BRSET_16266/1.0.0/fundus_photos/img00001.jpg",
  "visual_acuity": {
    "right_eye": null,
    "left_eye": null
  },
  "lens_status": null,
  "prior_laser": null,
  "prior_anti_vegf": null,
  "hba1c": null,
  "diabetes_duration": 12.0,
  "prior_vitrectomy": null,
  "symptoms": [],
  "image_quality": "Adequate",
  "missing_fields": [
    "visual_acuity",
    "lens_status",
    "prior_laser",
    "prior_anti_vegf",
    "hba1c",
    "prior_vitrectomy",
    "symptoms"
  ]
}


In [17]:

# Basic contract validation

required_keys = {
    "image_id",
    "visual_acuity",
    "lens_status",
    "prior_laser",
    "prior_anti_vegf",
    "hba1c",
    "diabetes_duration",
    "prior_vitrectomy",
    "symptoms",
    "image_quality",
    "missing_fields",
}

missing_keys = required_keys - set(note_output.keys())

assert not missing_keys, f"Missing output keys: {missing_keys}"
assert isinstance(note_output["visual_acuity"], dict)
assert set(note_output["visual_acuity"].keys()) == {"right_eye", "left_eye"}
assert isinstance(note_output["symptoms"], list)
assert isinstance(note_output["missing_fields"], list)

print("Clinical-Note Agent contract: PASS")


Clinical-Note Agent contract: PASS


In [18]:

# Generate a small in-memory batch for inspection.
# Increase this later if needed; 50 is enough for a quick sanity check.

SAMPLE_SIZE = min(50, len(brset))

note_agent_outputs = [
    clinical_note_agent(record)
    for record in brset[:SAMPLE_SIZE]
]

print(f"Generated {len(note_agent_outputs)} Clinical-Note Agent outputs.")
print("First output image_id:", note_agent_outputs[0]["image_id"])
print("First output diabetes_duration:", note_agent_outputs[0]["diabetes_duration"])
print("First output image_quality:", note_agent_outputs[0]["image_quality"])


Generated 50 Clinical-Note Agent outputs.
First output image_id: /All_data/BRSET_16266/1.0.0/fundus_photos/img00001.jpg
First output diabetes_duration: 12.0
First output image_quality: Adequate
